In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import einops
import matplotlib.pyplot as plt
from collections.abc import Callable, Iterable
from typing import Optional
import math
import tqdm

import psutil

total_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"Total RAM: {total_gb:.2f} GB")

available_gb = psutil.virtual_memory().available / (1024 ** 3)
print(f"Available RAM: {available_gb:.2f} GB")






data_dir = "/scratch/shared/beegfs/piyush/datasets/text_data"
filepath = f"{data_dir}/TinyStoriesV2-GPT4-train-tokenized.npy"
assert os.path.exists(filepath)
data = np.load(filepath, mmap_mode='r')
type(data), len(data)

import os
import psutil

_proc = psutil.Process(os.getpid())

def get_rss_mb() -> float:
    """Current process RSS in MB."""
    return _proc.memory_info().rss / (1024 * 1024)


class MemoryTracker:
    """Tracks current and peak RSS."""
    def __init__(self):
        self.peak = get_rss_mb()

    def update(self) -> tuple[float, float]:
        current = get_rss_mb()
        self.peak = max(self.peak, current)
        return current, self.peak


import numpy as np

def iterate_with_memmap(path, chunk_size=100_000):
    arr = np.load(path, mmap_mode='r')
    tracker = MemoryTracker()

    pbar = tqdm.tqdm_notebook(range(0, len(arr), chunk_size))

    for i in pbar:
        chunk = arr[i:i + chunk_size]

        # ---- your processing here ----
        # Check: if any token is beyond vocab size
        assert not (chunk >= 10257).sum().astype(bool)

        # update memory stats
        current, peak = tracker.update()

        # add to tqdm bar
        pbar.set_postfix({
            "RAM": f"{current:.1f} MB",
            "Peak": f"{peak:.1f} MB"
        })

        del chunk  # optional (helps clarity, not always required)

    return tracker.peak


iterate_with_memmap(filepath)

In [35]:
chunk[0]

np.uint16(430)

In [29]:
from tqdm import tqdm, tqdm_notebook
chunk_size = 1000
vocab_size = 10257
for i in tqdm_notebook(range(0, data.shape[0], chunk_size)):
    chunk = data[i:i+chunk_size]
    # Check: if any token is beyond vocab size
    assert not (chunk >= 20157).sum().astype(bool)

/tmp/ipykernel_1028166/3857983406.py:4: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i in tqdm_notebook(range(0, data.shape[0], chunk_size)):


  0%|          | 0/541125 [00:00<?, ?it/s]

**Convert npz to npy for efficient loading**

In [16]:
data_dir = "/scratch/shared/beegfs/piyush/datasets/text_data"
filenames = ["TinyStoriesV2-GPT4-train-tokenized.npz",  "TinyStoriesV2-GPT4-valid-tokenized.npz",  "owt_train-tokenized.npz",  "owt_valid-tokenized.npz"]
for f in filenames:
    filepath = f"{data_dir}/{f}"
    assert os.path.exists(filepath)

    # Load npz file
    data = np.load(filepath)['ids']
    print(f"Filename: {f} | Number of tokens: {len(data)}")

    # Save as npy file
    np.save(os.path.join(data_dir, f.replace(".npz", ".npy")), data)

Filename: TinyStoriesV2-GPT4-train-tokenized.npz | Number of tokens: 541124931
Filename: TinyStoriesV2-GPT4-valid-tokenized.npz | Number of tokens: 5464803
Filename: owt_train-tokenized.npz | Number of tokens: 2727181568
Filename: owt_valid-tokenized.npz | Number of tokens: 66399684
